# Arabic / English Sentiment Classifier

This notebook runs the comparison on a local sample. The clean repository intentionally excludes source text and row-level outputs because upstream data may contain third-party content and numeric identifiers.

The sample is balanced and educational, not representative of production prevalence. XLM-R is used without fine-tuning; source/domain and pretrained-data overlap cannot be ruled out.

## 1. Mount Google Drive and set the project path
Upload this cleaned repository folder to Drive. Update `PROJECT_DIR` if you use a different location.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/Arabic_English_Sentiment_Classifier_GitHub_Ready_Clean')
assert PROJECT_DIR.exists(), f'Folder not found: {PROJECT_DIR}. Put the package in the expected Drive location.'
%cd $PROJECT_DIR

## 2. Install dependencies
The XLM-R weights are downloaded from Hugging Face at first use. A GPU runtime is optional but speeds up transformer inference.

In [ ]:
!pip -q install -r requirements.txt

## 3. Create a local sample (optional, large download)
The sample is not bundled. After reviewing `DATA_SOURCES.md` and the source terms, run the next cell to download about 0.9 GB of Parquet data and create the local sample. Do not commit or share generated text.

In [ ]:
!python src/prepare_dataset.py --output data/processed/bilingual_sentiment_sample.csv

In [ ]:
import pandas as pd
data = pd.read_csv('data/processed/bilingual_sentiment_sample.csv')
print(data.shape)
display(data.groupby(['split','language','label']).size().unstack(fill_value=0))

## 4. Train and compare both approaches
The classical model selects regularization on validation macro-F1. Both methods use the same held-out rows; XLM-R is applied without fine-tuning. Training creates local row-level outputs; keep them private and do not commit them.

In [ ]:
!python src/train_evaluate.py --data data/processed/bilingual_sentiment_sample.csv --output .

## 5. Review aggregate reports and confusion matrices
This section displays aggregate metrics and charts only; it does not open raw-text error examples.

In [ ]:
display(pd.read_csv('reports/model_comparison.csv'))
display(pd.read_csv('reports/per_language_metrics.csv'))
from IPython.display import Image, display as show
show(Image('reports/figures/macro_f1_model_comparison.png'))
show(Image('reports/figures/macro_f1_by_language.png'))
show(Image('reports/figures/confusion_tfidf.png'))
show(Image('reports/figures/confusion_xlmr.png'))

## 6. Try your own illustrative text
These generic examples are demonstrations, not records sampled from the dataset.

In [ ]:
!python src/predict.py --model classical --language ar --text 'الخدمة ممتازة وسريعة'
!python src/predict.py --model classical --language en --text 'The support team solved my issue quickly'

## Limitations
This is a small, balanced educational sample from a multi-source, multi-domain dataset. It is not representative of live prevalence. Sentiment models can fail on sarcasm, dialect, code-switching, negation, and domain-specific language. See `DATA_SOURCES.md` for privacy and redistribution limits.